# Colombo Stock Exchange (CSE) 30-Day Return Forecasting
## 01: Exploratory Data Analysis & Purged Walk-Forward Cross-Validation (OHLCV Only)

This notebook demonstrates:
1. **Stock Ingestion & OHLCV Standardization**: Loading historical OHLCV data from `data/raw/`.
2. **Technical Feature Engineering**: Deriving RSI(14), MACD, EMAs, Parkinson Volatility, Bollinger Bands, and forward 30-day cumulative log return targets purely from price and volume.
3. **Purged Walk-Forward Cross-Validation**: Leak-free evaluation of multi-quantile LightGBM models with a 30-day embargo gap.
4. **Dual-Layer Explainability**: TreeSHAP technical attributions and retail heuristic badges.

In [ ]:
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Ensure project root is in path
sys.path.append(os.path.abspath('..'))

from src.ingestion.data_loader import CSEDataLoader
from src.features.technical_features import TechnicalFeatureEngine
from src.modeling.walk_forward_cv import PurgedWalkForwardCV
from src.modeling.train_lightgbm import MultiQuantileLGBMTrainer
from src.explainability.shap_service import TreeSHAPService
from src.explainability.heuristics_engine import HeuristicsEngine

print("CSE Stock Intelligence Pipeline loaded successfully.")

### 1. Ingestion & Historical OHLCV Loading

In [ ]:
loader = CSEDataLoader(raw_dir='../data/raw')
df_jkh = loader.load_stock_csv('JKH')

print(f"Loaded JKH historical bars: {len(df_jkh)} rows from {df_jkh['date'].min().date()} to {df_jkh['date'].max().date()}")
df_jkh.tail()

### 2. Feature Engineering & Target Formulation (OHLCV Dynamics Only)

Target: Forward 30-trading-day cumulative log return:
$$y_t = \ln\left(\frac{P_{t+30}}{P_t}\right)$$

In [ ]:
feature_engine = TechnicalFeatureEngine(target_horizon=30)
full_df = feature_engine.transform(df_jkh)

print(f"Engineered technical feature matrix shape: {full_df.shape}")
full_df[['date', 'close', 'rsi', 'parkinson_vol', 'bb_pct_b', 'price_to_sma_60', 'target_log_return_30d']].tail()

### 3. Purged & Embargoed Walk-Forward Cross-Validation

In [ ]:
cv = PurgedWalkForwardCV(n_splits=5, test_window=30, embargo_window=30, min_train_size=250)
splits = cv.get_split_dates(full_df.dropna(subset=['target_log_return_30d']))
pd.DataFrame(splits)

### 4. Multi-Quantile LightGBM Training (P10, P50, P90)

In [ ]:
from src.modeling.predict import FEATURE_COLUMNS
candidate_features = [c for c in FEATURE_COLUMNS if c in full_df.columns]

trainer = MultiQuantileLGBMTrainer(quantiles=[0.1, 0.5, 0.9])
eval_results = trainer.evaluate_walk_forward(full_df, candidate_features, target_col='target_log_return_30d', cv_engine=cv)

print("Walk-Forward Evaluation Summary:")
print(f"Overall Median MAE: {eval_results.get('overall_median_mae')}")
print(f"Empirical 80% Coverage (P10-P90): {eval_results.get('overall_coverage_p10_p90'):.2%}")
pd.DataFrame(eval_results['folds'])

### 5. Dual-Layer Explainability: TreeSHAP & Technical Heuristics

In [ ]:
models = trainer.fit_final_models(full_df, candidate_features)
median_model = models[0.5]

shap_service = TreeSHAPService(median_model, candidate_features)
shap_summary = shap_service.explain_prediction(full_df.tail(1))

print(f"Predicted 30-Day Return: {shap_summary['predicted_log_return']:+.4f}")
print(f"Base Value (Historical Mean): {shap_summary['base_value_return']:+.4f}")
print("\nTop Positive Drivers:")
display(pd.DataFrame(shap_summary['top_positive_drivers']))

print("\nRetail Heuristic Badges:")
heuristics = HeuristicsEngine()
badges = heuristics.evaluate_signals(full_df.iloc[-1], forecast_direction='bullish' if shap_summary['predicted_log_return'] > 0 else 'bearish')
display(pd.DataFrame(badges))